# 00 — Intro and Setup

Welcome! This project has you replicate and extend Anthropic's **Persona Vectors**
paper ([arXiv:2507.21509](https://arxiv.org/abs/2507.21509)). In short:

> Character traits like "evil", "sycophancy", and "hallucinate" correspond to
> linear directions in a chat model's activation space. If you can *extract* those
> directions, you can (a) monitor them during deployment, (b) causally steer the
> model toward or away from them, (c) predict how finetuning will shift them, and
> (d) flag training data that would induce them.

This notebook has three goals:

1. Verify your environment (GPU, OpenRouter key, model download).
2. Meet the `persona_lib` API you'll use throughout the course.
3. Skim the paper and orient on the motivating examples.


## Real-world motivations

A few incidents from the paper's intro:

- **Bing / Sydney (2023):** Microsoft's Bing chatbot sometimes slipped into a
  threatening, manipulative persona ("Sydney").
- **Grok (2025):** xAI's Grok briefly praised Hitler after modifications to its
  system prompt.
- **GPT-4o (April 2025):** An RLHF update made the model overly sycophantic — it
  validated harmful behaviors and reinforced negative emotions.

These are not one-off glitches; they are persona-level drifts. Being able to
*point at a trait* as a vector makes them tractable to study.


## Environment check


In [ ]:
import os, torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")
assert "OPENROUTER_API_KEY" in os.environ, "Set OPENROUTER_API_KEY before continuing."
print("OpenRouter model:", os.environ.get("OPENROUTER_MODEL", "anthropic/claude-haiku-4.5(default)"))


## Meet the library

Everything you'll import lives on `persona_lib`. You never import `torch` or a
tokenizer directly — if you feel the urge to register a forward hook, you're off
the rails.


In [ ]:
from persona_lib import (
    ActivationModel, ChatPrompt, steering,
    run_extraction, save_extraction, load_extraction,
    score_trait, score_coherence,
    list_traits, load_trait,
)
print("available traits on disk:", list_traits())


### A tiny model warm-up

We'll use Qwen2.5-7B-Instruct in real experiments, but while this notebook is
just validating the environment, load the 1.5B variant (fast, same API).


In [ ]:
model = ActivationModel("Qwen/Qwen2.5-1.5B-Instruct")
print("num_layers:", model.num_layers)
print("hidden_dim:", model.hidden_dim)

prompt = ChatPrompt(user="Give me a one-sentence fun fact about octopuses.")
print("\nbaseline output:", model.generate(prompt, max_new_tokens=50, temperature=0.7)[0])


### Activations

Every `ActivationModel` gives you residual-stream activations as a NumPy array:


In [ ]:
acts = model.collect_activations(prompt, position='last_prompt')
print("shape (n_prompts, n_layers, hidden_dim):", acts.shape)


## Paper reading — Section 1 + Figure 1 + Figure 2

Read the paper's Introduction (Section 1) and study **Figure 1** and **Figure 2**
before moving to notebook 01. In particular:

- **Figure 2** shows the extraction pipeline: contrastive system prompts →
  responses → activations → **difference of means** → persona vector.
- **Figure 1** shows the four downstream applications you'll build in notebooks
  02–06.


## What's next

- `01_extraction.ipynb` — compute your first persona vector.
- `02_monitoring.ipynb` — project activations to detect traits pre-generation.
- `03_inference_steering.ipynb` — prove the vector is causal.
- `04_finetune_shift.ipynb` … `07_capstone.ipynb` — the downstream suite.
